In [ ]:
"""
Análisis Tecno-Económico del Agregador en España
Modelo Conceptual de Ahorros Compartidos (Shared Savings) y Frontera de Pareto

Este script genera una representación teórica del modelo de negocio de un agregador
independiente. Ilustra la Frontera de Pareto que define el equilibrio entre el 
beneficio operativo del agregador y el ahorro neto del prosumidor, destacando la 
zona de viabilidad tecno-económica sujeta a restricciones de racionalidad individual.

Autor: Alberto Zafra Muñoz
"""

import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple

# ==========================================
# 1. MODELADO MATEMÁTICO DEL EQUILIBRIO
# ==========================================
def calcular_frontera_pareto(
    valor_total_vpp: float, 
    ahorro_base_uem: float, 
    ponderacion_alpha: float = 0.5
) -> Tuple[np.ndarray, np.ndarray, float, float]:
    """
    Calcula los vectores de la Frontera de Pareto y las coordenadas del punto óptimo.

    Parámetros:
        valor_total_vpp (float): Valor económico total diario generado por la co-optimización (€/día).
        ahorro_base_uem (float): Ahorro diario que el prosumidor ya obtenía de forma pasiva (Línea Base UEM).
        ponderacion_alpha (float): Factor de reparto del valor añadido (0.0 a 1.0). 
                                   0.5 representa un reparto equitativo del excedente.

    Retorna:
        Tuple: (vector ahorro_usuario, vector beneficio_agregador, óptimo_usuario, óptimo_agregador)
    """
    # Vectorización del ahorro del usuario (Eje de abscisas)
    ahorro_usuario = np.linspace(0, valor_total_vpp + 5, 200)

    # El beneficio del agregador es el valor residual tras compensar al usuario (Restricción lineal)
    beneficio_agregador = valor_total_vpp - ahorro_usuario

    # Cálculo del punto de equilibrio (Nash / Óptimo Teórico) basado en el parámetro alfa
    valor_anadido_vpp = valor_total_vpp - ahorro_base_uem
    
    # El usuario recupera su base y se lleva una fracción (1 - alpha) del valor añadido
    optimo_usuario = ahorro_base_uem + (valor_anadido_vpp * (1 - ponderacion_alpha))
    
    # El agregador se queda con la fracción (alpha) del valor añadido
    optimo_agregador = valor_anadido_vpp * ponderacion_alpha

    return ahorro_usuario, beneficio_agregador, optimo_usuario, optimo_agregador

# ==========================================
# 2. REPRESENTACIÓN GRÁFICA ACADÉMICA
# ==========================================
def visualizar_shared_savings(
    ahorro_usr: np.ndarray, 
    ben_agg: np.ndarray, 
    opt_usr: float, 
    opt_agg: float, 
    base_uem: float
):
    """
    Genera la figura de la Frontera de Pareto con áreas de viabilidad acotadas.
    """
    plt.figure(figsize=(10, 6))
    plt.style.use('seaborn-v0_8-whitegrid')

    # 1. Trazado de la Frontera de Pareto
    plt.plot(ahorro_usr, ben_agg, color='#1f77b4', linewidth=3,
             label='Frontera de Pareto (Valor Total Generado por la VPP)')

    # 2. Delimitación de la Zona de Viabilidad (Win-Win)
    # Restricción 1: El usuario debe ahorrar más que operando en solitario (Racionalidad Individual)
    # Restricción 2: El agregador debe tener beneficio operativo positivo
    plt.fill_between(
        ahorro_usr, 0, ben_agg,
        where=(ahorro_usr >= base_uem) & (ben_agg >= 0),
        color='#2ca02c', alpha=0.3, label='Zona de Viabilidad Tecno-Económica ("Win-Win")'
    )

    # 3. Líneas de Restricción del Sistema
    plt.axvline(x=base_uem, color='#d62728', linestyle='--', linewidth=2,
                label='Restricción de Racionalidad Individual (Línea Base UEM)')
    plt.axhline(y=0, color='black', linewidth=1.2, label='Umbral de Rentabilidad del Agregador')
    plt.axvline(x=0, color='black', linewidth=1)

    # 4. Marcador del Punto de Equilibrio Operativo
    plt.plot(opt_usr, opt_agg, marker='o', markersize=10, color='#d62728', zorder=5)
    plt.annotate(
        'Punto de Equilibrio Teórico\n(Ponderación $\\alpha$)',
        xy=(opt_usr, opt_agg),
        xytext=(opt_usr + 1.5, opt_agg + 3.5),
        arrowprops=dict(facecolor='#333333', shrink=0.05, width=1.5, headwidth=8),
        fontsize=11, fontweight='bold', color='#333333'
    )

    # 5. Formateo de ejes y etiquetas
    plt.title('Modelo Conceptual de Ahorros Compartidos (Shared Savings)', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Ahorro Neto en la Factura del Prosumidor (€/día)', fontsize=12)
    plt.ylabel('Beneficio Operativo Neto del Agregador (€/día)', fontsize=12)
    
    # Acotación de ejes dinámicos en base a los datos
    lim_x = ahorro_usr.max() - 5
    plt.xlim(0, lim_x)
    plt.ylim(-5, lim_x)

    plt.legend(loc='upper right', frameon=True, shadow=True, fontsize=10)
    plt.tight_layout()
    plt.show()

# ==========================================
# 3. BLOQUE PRINCIPAL DE EJECUCIÓN
# ==========================================
if __name__ == "__main__":
    # Parámetros del caso de estudio conceptual
    VALOR_TOTAL_VPP = 25.0       # Valor total diario generado por la optimización algorítmica (€)
    AHORRO_BASE_UEM = 8.0        # Ahorro que ya obtenía el usuario gestionando su propio excedente (€)
    FACTOR_REPARTO_ALPHA = 0.5   # Ponderación equitativa (50/50) del valor extra generado

    print("Calculando equilibrio microeconómico de la Frontera de Pareto...")
    x_ahorro, y_beneficio, pt_usr, pt_agg = calcular_frontera_pareto(
        VALOR_TOTAL_VPP, 
        AHORRO_BASE_UEM, 
        FACTOR_REPARTO_ALPHA
    )
    
    print("Generando visualización gráfica...")
    visualizar_shared_savings(x_ahorro, y_beneficio, pt_usr, pt_agg, AHORRO_BASE_UEM)